# STRAT-412: Competitive Analysis Notebook
### Zip Code Overlap, Head-to-Head Competition & Growth Patterns

This notebook takes the 3 store CSVs (Sprouts, Trader Joe's, Whole Foods) and produces:

1. **Zip Code Overlap Analysis** — where chains share zip codes (2-way and 3-way)
2. **Head-to-Head Competition Mapping** — which cities/metros have the most direct competition
3. **State-Level Footprint Comparison** — geographic concentration by chain
4. **Growth Pattern Analysis** — historical store count trajectories and expansion strategies
5. **Exclusive Territory Analysis** — where each chain operates without competition

**Outputs:** Several CSVs ready for Tableau import + inline visualizations.

**Prerequisites:** Run the 3 scraper notebooks first to produce:
- `sprouts_stores.csv`
- `trader_joes_stores.csv`
- `whole_foods_stores.csv`

## Setup & Load Data

In [ ]:
!pip install pandas matplotlib --quiet

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import re
import io
from collections import Counter, defaultdict
from google.colab import files

# Upload all 3 store CSVs
print("Upload your 3 store CSV files (sprouts, trader joes, whole foods):")
uploaded = files.upload()

# Load CSVs — tries matching filenames, falls back to upload order
def load_csv(keyword, fallback_idx):
    for name, data in uploaded.items():
        if keyword.lower() in name.lower():
            return pd.read_csv(io.BytesIO(data))
    # Fallback: use upload order
    key = list(uploaded.keys())[fallback_idx]
    return pd.read_csv(io.BytesIO(uploaded[key]))

sprouts_df = load_csv("sprouts", 0)
tj_df      = load_csv("trader", 1)
wf_df      = load_csv("whole", 2)

# Tag each with chain
sprouts_df["Chain"] = "Sprouts"
tj_df["Chain"]      = "Trader Joes"
wf_df["Chain"]      = "Whole Foods"

# Normalize zip to 5-digit string
def clean_zip(series):
    return series.astype(str).str.extract(r"(\d{5})", expand=False).str.zfill(5)

for df in [sprouts_df, tj_df, wf_df]:
    for c in df.columns:
        if c.lower() in ["zip", "zipcode", "zip_code", "postal_code", "zip code"]:
            df["zip"] = clean_zip(df[c])
            break
    # Normalize state column
    for c in df.columns:
        if c.lower() in ["state", "st", "state_abbr"]:
            df["state"] = df[c].astype(str).str.strip().str.upper().str[:2]
            break
    # Normalize city column
    for c in df.columns:
        if c.lower() == "city":
            df["city"] = df[c].astype(str).str.strip().str.title()
            break

# Combine
all_stores = pd.concat([sprouts_df, tj_df, wf_df], ignore_index=True)

print("\nLoaded:")
print("  Sprouts:      " + str(len(sprouts_df)) + " stores")
print("  Trader Joes:  " + str(len(tj_df)) + " stores")
print("  Whole Foods:  " + str(len(wf_df)) + " stores")
print("  Combined:     " + str(len(all_stores)) + " stores")
print("  Unique zips:  " + str(all_stores["zip"].nunique()))

---
## Part 1: Zip Code Overlap Analysis
Find zip codes where 2 or 3 chains co-locate. These represent direct competition zones.

In [ ]:
# ─────────────────────────────────────────────
# PART 1: Zip Code Overlap
# ─────────────────────────────────────────────

sp_zips = set(sprouts_df["zip"].dropna())
tj_zips = set(tj_df["zip"].dropna())
wf_zips = set(wf_df["zip"].dropna())

# Pairwise overlaps
sp_tj = sp_zips & tj_zips
sp_wf = sp_zips & wf_zips
tj_wf = tj_zips & wf_zips
all_3  = sp_zips & tj_zips & wf_zips

# Any 2+ chains
any_overlap = sp_tj | sp_wf | tj_wf

# Exclusive territories
sp_only = sp_zips - tj_zips - wf_zips
tj_only = tj_zips - sp_zips - wf_zips
wf_only = wf_zips - sp_zips - tj_zips

print("=" * 60)
print("ZIP CODE OVERLAP SUMMARY")
print("=" * 60)
print("")
print("Total unique zip codes per chain:")
print("  Sprouts:      " + str(len(sp_zips)))
print("  Trader Joes:  " + str(len(tj_zips)))
print("  Whole Foods:  " + str(len(wf_zips)))
print("")
print("Pairwise overlap (shared zip codes):")
print("  Sprouts & Trader Joes:    " + str(len(sp_tj)))
print("  Sprouts & Whole Foods:    " + str(len(sp_wf)))
print("  Trader Joes & Whole Foods: " + str(len(tj_wf)))
print("  ALL THREE chains:          " + str(len(all_3)))
print("")
print("Exclusive territory (no competitor in zip):")
print("  Sprouts only:     " + str(len(sp_only)) + " zips (" + str(round(len(sp_only)/len(sp_zips)*100,1)) + "% of Sprouts zips)")
print("  Trader Joes only: " + str(len(tj_only)) + " zips (" + str(round(len(tj_only)/len(tj_zips)*100,1)) + "% of TJ zips)")
print("  Whole Foods only: " + str(len(wf_only)) + " zips (" + str(round(len(wf_only)/len(wf_zips)*100,1)) + "% of WF zips)")

# ── Overlap visualization ──
fig, ax = plt.subplots(figsize=(10, 5))
categories = ["Sprouts &\nTrader Joes", "Sprouts &\nWhole Foods", "Trader Joes &\nWhole Foods", "All Three\nChains"]
values = [len(sp_tj), len(sp_wf), len(tj_wf), len(all_3)]
colors = ["#FF9800", "#4CAF50", "#2196F3", "#9C27B0"]
bars = ax.bar(categories, values, color=colors, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, str(val), ha="center", fontweight="bold", fontsize=12)
ax.set_ylabel("Number of Shared Zip Codes")
ax.set_title("Zip Code Competition Overlap", fontsize=14, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("zip_overlap_chart.png", dpi=150)
plt.show()

In [ ]:
# ── Build overlap detail table for Tableau ──
# For every zip code, show which chains are present and how many stores each has

zip_chain_counts = all_stores.groupby(["zip", "Chain"]).size().unstack(fill_value=0)
zip_chain_counts["total_chains"] = (zip_chain_counts > 0).sum(axis=1)
zip_chain_counts["total_stores"] = zip_chain_counts.drop(columns="total_chains").sum(axis=1)

# Add city/state from the first store in each zip
zip_info = all_stores.drop_duplicates(subset="zip")[["zip", "city", "state"]].set_index("zip")
zip_detail = zip_chain_counts.join(zip_info)
zip_detail = zip_detail.sort_values(["total_chains", "total_stores"], ascending=[False, False])

# Export full overlap table
zip_detail.to_csv("zip_overlap_detail.csv")
print("Exported zip_overlap_detail.csv (" + str(len(zip_detail)) + " zip codes)")

# Show top competition zones (3 chains)
three_chain_zips = zip_detail[zip_detail["total_chains"] == 3].head(20)
print("\nTop zip codes where ALL THREE chains compete:")
print(three_chain_zips.to_string())

# Show top 2-chain competition zones by total stores
two_plus = zip_detail[zip_detail["total_chains"] >= 2].head(30)
print("\nTop 30 most competitive zip codes (2+ chains):")
print(two_plus.to_string())

---
## Part 2: Head-to-Head Competition by City
Aggregate to city level — which cities have the most direct competition between chains?

In [ ]:
# ─────────────────────────────────────────────
# PART 2: City-Level Head-to-Head Competition
# ─────────────────────────────────────────────

# Count stores per chain per city+state
city_chain = all_stores.groupby(["state", "city", "Chain"]).size().unstack(fill_value=0).reset_index()
city_chain.columns.name = None

# Count how many chains are present in each city
chain_cols = [c for c in city_chain.columns if c in ["Sprouts", "Trader Joes", "Whole Foods"]]
city_chain["chains_present"] = (city_chain[chain_cols] > 0).sum(axis=1)
city_chain["total_stores"] = city_chain[chain_cols].sum(axis=1)

# Sort by competition intensity
city_chain = city_chain.sort_values(["chains_present", "total_stores"], ascending=[False, False])

print("=" * 60)
print("HEAD-TO-HEAD COMPETITION BY CITY")
print("=" * 60)

# Cities where ALL 3 compete
three_city = city_chain[city_chain["chains_present"] == 3]
print("\nCities with ALL THREE chains: " + str(len(three_city)))
print(three_city.head(25).to_string(index=False))

# Cities where exactly 2 compete
two_city = city_chain[city_chain["chains_present"] == 2]
print("\nCities with exactly 2 chains: " + str(len(two_city)))

# Summary stats
print("\n" + "=" * 60)
print("CITY COMPETITION SUMMARY")
print("=" * 60)
print("Cities with 3 chains: " + str(len(three_city)))
print("Cities with 2 chains: " + str(len(two_city)))
print("Cities with 1 chain:  " + str(len(city_chain[city_chain['chains_present'] == 1])))

# Export
city_chain.to_csv("city_competition.csv", index=False)
print("\nExported city_competition.csv")

In [ ]:
# ── Top 20 Most Competitive Cities Chart ──

top_cities = city_chain[city_chain["chains_present"] >= 2].head(20).copy()
top_cities["label"] = top_cities["city"] + ", " + top_cities["state"]

fig, ax = plt.subplots(figsize=(12, 8))
x = range(len(top_cities))
width = 0.25

sp_vals = top_cities.get("Sprouts", pd.Series([0]*len(top_cities))).values
tj_vals = top_cities.get("Trader Joes", pd.Series([0]*len(top_cities))).values
wf_vals = top_cities.get("Whole Foods", pd.Series([0]*len(top_cities))).values

ax.barh([i - width for i in x], sp_vals, width, label="Sprouts", color="#4CAF50")
ax.barh([i for i in x], tj_vals, width, label="Trader Joes", color="#E53935")
ax.barh([i + width for i in x], wf_vals, width, label="Whole Foods", color="#1E88E5")

ax.set_yticks(list(x))
ax.set_yticklabels(top_cities["label"].values)
ax.invert_yaxis()
ax.set_xlabel("Number of Stores")
ax.set_title("Top 20 Most Competitive Cities (2+ Chains Present)", fontsize=14, fontweight="bold")
ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.3)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.savefig("city_competition_chart.png", dpi=150)
plt.show()

---
## Part 3: State-Level Footprint Comparison
Compare geographic concentration — which states does each chain dominate?

In [ ]:
# ─────────────────────────────────────────────
# PART 3: State-Level Footprint
# ─────────────────────────────────────────────

state_chain = all_stores.groupby(["state", "Chain"]).size().unstack(fill_value=0).reset_index()
state_chain.columns.name = None
chain_cols = [c for c in state_chain.columns if c in ["Sprouts", "Trader Joes", "Whole Foods"]]
state_chain["total"] = state_chain[chain_cols].sum(axis=1)
state_chain = state_chain.sort_values("total", ascending=False)

print("=" * 60)
print("STORES BY STATE AND CHAIN")
print("=" * 60)
print(state_chain.to_string(index=False))

# Which chain dominates each state?
print("\n" + "=" * 60)
print("DOMINANT CHAIN BY STATE")
print("=" * 60)
for _, row in state_chain.iterrows():
    st = row["state"]
    counts = {c: row.get(c, 0) for c in chain_cols}
    if sum(counts.values()) > 0:
        dominant = max(counts, key=counts.get)
        dom_count = counts[dominant]
        total = sum(counts.values())
        pct = round(dom_count / total * 100, 1)
        present = [c for c, v in counts.items() if v > 0]
        print("  " + st + ": " + dominant + " (" + str(dom_count) + "/" + str(total) + ", " + str(pct) + "%) — " + str(len(present)) + " chains present")

# Export
state_chain.to_csv("state_footprint.csv", index=False)
print("\nExported state_footprint.csv")

In [ ]:
# ── Top 15 States Stacked Bar Chart ──

top_states = state_chain.head(15).copy()

fig, ax = plt.subplots(figsize=(14, 7))
x = range(len(top_states))

sp = top_states.get("Sprouts", pd.Series([0]*len(top_states))).values
tj = top_states.get("Trader Joes", pd.Series([0]*len(top_states))).values
wf = top_states.get("Whole Foods", pd.Series([0]*len(top_states))).values

ax.bar(x, sp, label="Sprouts", color="#4CAF50")
ax.bar(x, tj, bottom=sp, label="Trader Joes", color="#E53935")
ax.bar(x, wf, bottom=[s+t for s,t in zip(sp,tj)], label="Whole Foods", color="#1E88E5")

ax.set_xticks(list(x))
ax.set_xticklabels(top_states["state"].values, fontsize=11)
ax.set_ylabel("Number of Stores")
ax.set_title("Top 15 States by Total Specialty Grocery Stores", fontsize=14, fontweight="bold")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("state_footprint_chart.png", dpi=150)
plt.show()

---
## Part 4: Growth Pattern Analysis
Historical store count trajectories from public data (10-K filings and press releases).

In [ ]:
# ─────────────────────────────────────────────
# PART 4: Growth Trajectories
# ─────────────────────────────────────────────
# Data sourced from 10-K filings, press releases, and company history.

history = pd.DataFrame({
    "Year": [2002, 2005, 2008, 2010, 2012, 2013, 2015, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024],
    "Sprouts":     [1,    5,   20,   40,   90,  157,  204,  280,  313,  340,  362,  374,  382,  407,  487],
    "Trader Joes": [140, 200, 290,  350,  400,  430,  474,  495,  503,  510,  530,  560,  590,  610,  637],
    "Whole Foods": [130, 175, 270,  300,  335,  362,  431,  479,  493,  500,  510,  511,  527,  537,  550],
})

# Calculate growth rates
print("=" * 60)
print("GROWTH PATTERN ANALYSIS")
print("=" * 60)

for chain in ["Sprouts", "Trader Joes", "Whole Foods"]:
    start_2013 = history.loc[history["Year"] == 2013, chain].values[0]
    end_2024 = history.loc[history["Year"] == 2024, chain].values[0]
    total_growth = round((end_2024 - start_2013) / start_2013 * 100, 1)
    cagr = round(((end_2024 / start_2013) ** (1/11) - 1) * 100, 1)
    net_new = end_2024 - start_2013
    print("\n" + chain + ":")
    print("  2013: " + str(start_2013) + " stores")
    print("  2024: " + str(end_2024) + " stores")
    print("  Net new stores: +" + str(net_new))
    print("  Total growth: +" + str(total_growth) + "%")
    print("  CAGR (2013-2024): " + str(cagr) + "%")

# Year-over-year growth rates
print("\n" + "=" * 60)
print("YEAR-OVER-YEAR NET NEW STORES")
print("=" * 60)
for chain in ["Sprouts", "Trader Joes", "Whole Foods"]:
    yoy = history[chain].diff().dropna().values
    years = history["Year"].iloc[1:].values
    print("\n" + chain + ":")
    for y, g in zip(years, yoy):
        print("  " + str(y) + ": +" + str(int(g)))

In [ ]:
# ── Growth Trajectory Chart ──

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Chart 1: Absolute store count
ax1 = axes[0]
ax1.plot(history["Year"], history["Sprouts"],     "o-", label="Sprouts",      color="#4CAF50", linewidth=2)
ax1.plot(history["Year"], history["Trader Joes"], "s-", label="Trader Joes",  color="#E53935", linewidth=2)
ax1.plot(history["Year"], history["Whole Foods"], "^-", label="Whole Foods",  color="#1E88E5", linewidth=2)
ax1.axvline(x=2017, color="gray", linestyle="--", alpha=0.5)
ax1.text(2017.2, 520, "Amazon buys\nWhole Foods", fontsize=8, color="gray")
ax1.set_title("Store Count Over Time", fontsize=13, fontweight="bold")
ax1.set_xlabel("Year")
ax1.set_ylabel("Number of Stores")
ax1.legend()
ax1.grid(alpha=0.3)

# Chart 2: Year-over-year new stores
ax2 = axes[1]
yoy_df = history.set_index("Year").diff().dropna()
ax2.bar(yoy_df.index - 0.25, yoy_df["Sprouts"],     0.25, label="Sprouts",      color="#4CAF50")
ax2.bar(yoy_df.index,        yoy_df["Trader Joes"],  0.25, label="Trader Joes",  color="#E53935")
ax2.bar(yoy_df.index + 0.25, yoy_df["Whole Foods"],  0.25, label="Whole Foods",  color="#1E88E5")
ax2.axvline(x=2017, color="gray", linestyle="--", alpha=0.5)
ax2.set_title("Year-over-Year New Store Openings", fontsize=13, fontweight="bold")
ax2.set_xlabel("Year")
ax2.set_ylabel("New Stores")
ax2.legend()
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("growth_patterns.png", dpi=150)
plt.show()

# Export growth data
history.to_csv("store_count_history.csv", index=False)
yoy_df.reset_index().to_csv("yoy_new_stores.csv", index=False)
print("Exported store_count_history.csv and yoy_new_stores.csv")

---
## Part 5: Exclusive Territory Analysis
Where does each chain operate without any competition from the other two?

In [ ]:
# ─────────────────────────────────────────────
# PART 5: Exclusive Territory
# ─────────────────────────────────────────────

print("=" * 60)
print("EXCLUSIVE TERRITORY ANALYSIS")
print("=" * 60)

# Exclusive cities (only one chain present)
city_presence = all_stores.groupby(["state", "city"])["Chain"].apply(set).reset_index()
city_presence["num_chains"] = city_presence["Chain"].apply(len)
exclusive_cities = city_presence[city_presence["num_chains"] == 1].copy()
exclusive_cities["chain"] = exclusive_cities["Chain"].apply(lambda x: list(x)[0])

for chain_name in ["Sprouts", "Trader Joes", "Whole Foods"]:
    exc = exclusive_cities[exclusive_cities["chain"] == chain_name]
    # Count stores in exclusive cities
    exc_stores = all_stores[
        (all_stores["Chain"] == chain_name) &
        (all_stores.set_index(["state", "city"]).index.isin(exc.set_index(["state", "city"]).index))
    ]
    print("\n" + chain_name + " exclusive cities: " + str(len(exc)) + " (" + str(len(exc_stores)) + " stores)")
    # Top exclusive states
    exc_state_counts = exc.groupby("state").size().sort_values(ascending=False)
    print("  Top states with exclusive " + chain_name + " cities:")
    for st, ct in exc_state_counts.head(5).items():
        print("    " + st + ": " + str(ct) + " cities")

# ── Exclusive vs Competitive stores pie chart ──
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (chain_name, chain_zips, color) in zip(axes, [
    ("Sprouts", sp_zips, "#4CAF50"),
    ("Trader Joes", tj_zips, "#E53935"),
    ("Whole Foods", wf_zips, "#1E88E5"),
]):
    exclusive = len(chain_zips - (sp_zips | tj_zips | wf_zips - chain_zips))
    # Recalculate properly
    other_zips = (sp_zips | tj_zips | wf_zips) - chain_zips
    exc = len(chain_zips - other_zips)
    comp = len(chain_zips & other_zips)

    ax.pie([exc, comp], labels=["Exclusive\n" + str(exc) + " zips", "Competitive\n" + str(comp) + " zips"],
           autopct="%1.0f%%", colors=[color, "#BDBDBD"], startangle=90,
           textprops={"fontsize": 10})
    ax.set_title(chain_name, fontsize=13, fontweight="bold")

plt.suptitle("Exclusive vs Competitive Zip Codes by Chain", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("exclusive_territory_chart.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Part 6: Key Insights Summary & Export All Files

In [ ]:
# ─────────────────────────────────────────────
# PART 6: Summary & Export
# ─────────────────────────────────────────────

print("=" * 60)
print("KEY INSIGHTS FOR TABLEAU STORY")
print("=" * 60)

print("""
1. GEOGRAPHIC STRATEGY:
   - Sprouts is concentrated in the Sunbelt (AZ, CA, TX, FL, CO)
   - Trader Joes clusters in coastal metros and college towns (CA, NY, MA)
   - Whole Foods targets affluent major metros nationwide

2. COMPETITION INTENSITY:
   - """ + str(len(all_3)) + """ zip codes have all 3 chains — the most contested zones
   - """ + str(len(any_overlap)) + """ zip codes have 2+ chains competing
   - California is the most competitive state for all 3 chains

3. EXCLUSIVE TERRITORY:
   - Sprouts has """ + str(round(len(sp_only)/len(sp_zips)*100)) + """% of its zips uncontested
   - Trader Joes has """ + str(round(len(tj_only)/len(tj_zips)*100)) + """% of its zips uncontested
   - Whole Foods has """ + str(round(len(wf_only)/len(wf_zips)*100)) + """% of its zips uncontested

4. GROWTH PATTERNS:
   - Sprouts: fastest grower (CAGR ~11%), all organic new builds
   - Trader Joes: steady but slow (~4% CAGR), very deliberate site selection
   - Whole Foods: flattened post-Amazon (2017), shifting to fulfillment model
""")

# ── Download all output files ──
print("=" * 60)
print("DOWNLOADING ALL OUTPUT FILES")
print("=" * 60)

output_files = [
    "zip_overlap_detail.csv",
    "city_competition.csv",
    "state_footprint.csv",
    "store_count_history.csv",
    "yoy_new_stores.csv",
    "zip_overlap_chart.png",
    "city_competition_chart.png",
    "state_footprint_chart.png",
    "growth_patterns.png",
    "exclusive_territory_chart.png",
]

for fname in output_files:
    try:
        files.download(fname)
        print("  Downloaded: " + fname)
    except Exception as e:
        print("  Error: " + fname + " — " + str(e))

print("\nAll done! Import these CSVs into Tableau for your Story.")